# CardioIA — Fase 2 · Parte 1
## Extração de sintomas e sugestão de diagnóstico

Este notebook lê relatos escritos em linguagem natural, encontra neles as expressões
de um **mapa de conhecimento** (uma ontologia sintoma → doença) e sugere uma hipótese
diagnóstica, mostrando a evidência que sustenta cada sugestão.

É deliberadamente um sistema **baseado em regras**, não em aprendizado: com 10 relatos
não há dado suficiente para treinar nada, e a regra tem uma vantagem que o modelo
estatístico da Parte 2 não terá — ela consegue dizer *por que* chegou àquela conclusão.

**Entradas**
| Arquivo | Conteúdo |
|---|---|
| `data/relatos_pacientes.txt` | 10 relatos de sintomas (fictícios, escritos por nós) |
| `data/mapa_conhecimento.csv` | 79 regras `sintoma_1 \| sintoma_2 \| doenca_associada`, 8 doenças |

A lógica vive em `src/extracao.py` — o notebook importa e demonstra, para que o mesmo
código rode também pela linha de comando (`python src/extracao.py`).

In [1]:
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path.cwd().parent))
from src.extracao import (
    carregar_mapa,
    carregar_relatos,
    encontrar_expressoes,
    normalizar,
    pontuar,
    relatorio,
)

pd.set_option("display.max_colwidth", 90)

---
## 1. Os relatos

Cada linha do `.txt` é um relato completo: **o que o paciente sente, quando começou e
como aquilo afeta a rotina** — os três elementos pedidos no enunciado. As frases são
inventadas por nós, mas o vocabulário e os quadros clínicos seguem os sinais descritos
na base da Fase 1 (`tipo_dor_toracica`, `angina_por_esforco`, `ecg_repouso`).

In [2]:
relatos = carregar_relatos()
print(f"{len(relatos)} relatos carregados\n")
for i, r in enumerate(relatos, 1):
    print(f"{i:>2}. {r}\n")

10 relatos carregados

 1. Ha dois dias estou com uma dor no peito que aperta e piora quando faco esforco fisico; ontem tive que parar no meio da subida para casa e a dor so passou depois de uns dez minutos sentado.

 2. Sinto um cansaco constante ha uma semana, mesmo depois de descansar, e minhas pernas estao inchadas no fim do dia a ponto de o sapato marcar; para dormir preciso de dois travesseiros senao me falta o ar deitado.

 3. Desde ontem a noite sinto o coracao disparado e batendo descompassado, como se falhasse uma batida, junto com uma tontura que me obriga a sentar; ja aconteceu tres vezes hoje enquanto eu estava parado.

 4. Tenho falta de ar para subir um lance de escada que eu subia sem problema ate o mes passado, e as vezes vem junto um aperto no peito que melhora quando eu paro de andar.

 5. Acordei hoje as cinco da manha com uma dor forte no peito que irradia para o braco esquerdo e para a mandibula, com suor frio e enjoo; nao passou com o remedio de estomago e ja faz

---
## 2. O mapa de conhecimento

Cada linha é uma **regra**: duas expressões que, juntas, apontam para uma doença.
Manter três colunas — `sintoma_1 | sintoma_2 | doenca_associada` — é o formato pedido
no enunciado, e ele carrega mais informação do que uma lista simples de palavras-chave:
o par codifica *combinação* de sinais, que é como o raciocínio clínico funciona.
"Dor no peito" sozinha não decide nada; "dor no peito + piora ao esforço" aponta para
angina, e "dor no peito + suor frio" para infarto.

> **Divergência assumida em relação ao enunciado.** O exemplo do enunciado mapeia
> `"falta de ar", "dificuldade para respirar" → Angina`. Clinicamente, dispneia isolada
> é sinal de **insuficiência cardíaca**; angina é definida pela dor/aperto torácico
> desencadeado por esforço. Mantivemos a classificação clínica correta e registramos a
> divergência aqui em vez de reproduzir o exemplo.

In [3]:
mapa = carregar_mapa()
print(f"{len(mapa)} regras · {mapa.doenca_associada.nunique()} doenças\n")
display(mapa.doenca_associada.value_counts().to_frame("regras"))
display(mapa[["sintoma_1", "sintoma_2", "doenca_associada"]].head(8))

79 regras · 8 doenças



,regras
doenca_associada,
AVC,13
Infarto agudo do miocardio,12
Insuficiencia cardiaca,12
Angina,10
Estenose aortica,10
Arritmia cardiaca,9
Hipertensao arterial,7
Pericardite,6


,sintoma_1,sintoma_2,doenca_associada
0,dor no peito,aperto no peito,Infarto agudo do miocardio
1,dor forte no peito,suor frio,Infarto agudo do miocardio
2,dor no peito,irradia para o braco esquerdo,Infarto agudo do miocardio
3,dor no peito,dor no braco esquerdo,Infarto agudo do miocardio
4,dor no peito,mandibula,Infarto agudo do miocardio
5,aperto no peito,suor frio,Infarto agudo do miocardio
6,dor no peito,enjoo,Infarto agudo do miocardio
7,dor no peito,nausea,Infarto agudo do miocardio


---
## 3. Normalização: o passo que faz o casamento funcionar

O paciente escreve "Não passou com o remédio", o mapa guarda "nao passou". Antes de
comparar qualquer coisa, texto e mapa passam pela mesma função: minúsculas, acentos
removidos, pontuação virando espaço. Sem isso, metade das regras nunca dispararia.

In [4]:
exemplo = "Acordei com uma DOR no peito — que irradia p/ o braço esquerdo!"
print("original: ", exemplo)
print("normalizado:", normalizar(exemplo))

original:  Acordei com uma DOR no peito — que irradia p/ o braço esquerdo!
normalizado: acordei com uma dor no peito que irradia p o braco esquerdo


---
## 4. Extração dos sintomas (com tratamento de negação)

`encontrar_expressoes` procura no relato todas as expressões do mapa. O detalhe que
evita o erro mais grosseiro: se a expressão vier logo depois de uma palavra de negação
(*não, sem, nunca, nenhum*), ela é descartada. "Não sinto dor no peito" não pode contar
como dor no peito — e é exatamente esse o erro que o classificador estatístico da
Parte 2 vai cometer.

In [5]:
expressoes = set(mapa.expr_1) | set(mapa.expr_2)

com    = "sinto dor no peito quando subo escada"
sem    = "nao sinto dor no peito, so vim renovar a receita"
print("com sintoma :", sorted(encontrar_expressoes(com, expressoes)))
print("com negação :", sorted(encontrar_expressoes(sem, expressoes)))

com sintoma : ['dor no peito']
com negação : []


---
## 5. Como a hipótese é escolhida

Para cada doença:

$$\text{pontuação} = \underbrace{|\text{expressões distintas encontradas}|}_{\text{amplitude da evidência}} + 2 \times \underbrace{|\text{regras com as duas expressões}|}_{\text{evidência combinada}}$$

Contar **expressões distintas**, e não linhas do mapa, corrige um viés que aparece
quando se soma linha a linha: "dor no peito" aparece em 7 regras de infarto, então uma
queixa genérica venceria por repetição, sem nenhuma evidência específica. O bônus de 2
pontos premia a regra que disparou inteira — que é a informação que o mapa realmente
codifica.

O sistema devolve também as **alternativas** e avisa quando há empate. Um sistema de
triagem que devolve uma única resposta sem margem esconde do médico exatamente a
informação de que ele precisa.

In [6]:
caso = relatos[0]
print(caso, "\n")
display(pd.DataFrame([
    {"doença": h.doenca, "pontuação": h.pontuacao,
     "regras completas": h.regras_completas, "sintomas encontrados": ", ".join(h.sintomas)}
    for h in pontuar(caso, mapa)
]))

Ha dois dias estou com uma dor no peito que aperta e piora quando faco esforco fisico; ontem tive que parar no meio da subida para casa e a dor so passou depois de uns dez minutos sentado. 



,doença,pontuação,regras completas,sintomas encontrados
0,Angina,5,1,"dor no peito, esforco fisico, piora quando faco esforco"
1,Estenose aortica,1,0,esforco fisico
2,Infarto agudo do miocardio,1,0,dor no peito
3,Pericardite,1,0,dor no peito


---
## 6. Os 10 relatos processados

In [7]:
for i, r in enumerate(relatos, 1):
    print(f"[Relato {i}] {r}")
    print(relatorio(r, mapa))
    print()

[Relato 1] Ha dois dias estou com uma dor no peito que aperta e piora quando faco esforco fisico; ontem tive que parar no meio da subida para casa e a dor so passou depois de uns dez minutos sentado.
  Sintomas extraidos: dor no peito, esforco fisico, piora quando faco esforco
  -> Hipotese: Angina (pontuacao 5, 1 regra(s) completa(s))
     Alternativas: Estenose aortica (1); Infarto agudo do miocardio (1)

[Relato 2] Sinto um cansaco constante ha uma semana, mesmo depois de descansar, e minhas pernas estao inchadas no fim do dia a ponto de o sapato marcar; para dormir preciso de dois travesseiros senao me falta o ar deitado.
  Sintomas extraidos: cansaco, cansaco constante, dois travesseiros, mesmo depois de descansar, sapato marcar
  -> Hipotese: Insuficiencia cardiaca (pontuacao 7, 1 regra(s) completa(s))
     Alternativas: Estenose aortica (1)

[Relato 3] Desde ontem a noite sinto o coracao disparado e batendo descompassado, como se falhasse uma batida, junto com uma tontura que me

---
## 7. Avaliação: o extrator acerta o que deveria?

"Funciona" precisa ser medido, não afirmado. Anotamos o diagnóstico que cada relato foi
escrito para representar (o gabarito) e comparamos com a saída do sistema.

**O que este número é e o que não é:** os relatos e o mapa foram escritos pela mesma
equipe, então 100% aqui significa "as regras cobrem os casos que nós mesmos criamos" —
consistência interna, não desempenho clínico. É o teto do que uma base de 10 frases
autorais pode demonstrar, e o registramos como tal.

In [8]:
gabarito = [
    "Angina", "Insuficiencia cardiaca", "Arritmia cardiaca", "Angina",
    "Infarto agudo do miocardio", "Hipertensao arterial", "AVC",
    "Pericardite", "Estenose aortica", "Insuficiencia cardiaca",
]

linhas = []
for r, esperado in zip(relatos, gabarito):
    ranking = pontuar(r, mapa)
    previsto = ranking[0].doenca if ranking else "(sem hipótese)"
    margem = ranking[0].pontuacao - ranking[1].pontuacao if len(ranking) > 1 else ranking[0].pontuacao
    linhas.append({"relato": r[:55] + "…", "esperado": esperado,
                   "sugerido": previsto, "margem p/ a 2ª": margem,
                   "ok": "✔" if previsto == esperado else "✘"})

resultado = pd.DataFrame(linhas)
display(resultado)
acertos = (resultado.ok == "✔").sum()
print(f"Acertos: {acertos}/{len(resultado)} ({acertos/len(resultado):.0%})")
print(f"Menor margem para a segunda hipótese: {resultado['margem p/ a 2ª'].min():g} ponto(s)")

,relato,esperado,sugerido,margem p/ a 2ª,ok
0,Ha dois dias estou com uma dor no peito que aperta e pi…,Angina,Angina,4,✔
1,"Sinto um cansaco constante ha uma semana, mesmo depois …",Insuficiencia cardiaca,Insuficiencia cardiaca,6,✔
2,Desde ontem a noite sinto o coracao disparado e batendo…,Arritmia cardiaca,Arritmia cardiaca,7,✔
3,Tenho falta de ar para subir um lance de escada que eu …,Angina,Angina,4,✔
4,Acordei hoje as cinco da manha com uma dor forte no pei…,Infarto agudo do miocardio,Infarto agudo do miocardio,10,✔
5,Ha tres meses sinto dor de cabeca latejante na nuca qua…,Hipertensao arterial,Hipertensao arterial,8,✔
6,Ontem de manha meu braco direito ficou dormente e fraco…,AVC,AVC,8,✔
7,Sinto uma dor no peito que piora quando eu respiro fund…,Pericardite,Pericardite,9,✔
8,Ha um mes tenho tido tontura e escurecimento da vista q…,Estenose aortica,Estenose aortica,10,✔
9,Tenho tosse seca a noite ha duas semanas e acordo sufoc…,Insuficiencia cardiaca,Insuficiencia cardiaca,6,✔


Acertos: 10/10 (100%)
Menor margem para a segunda hipótese: 4 ponto(s)


---
## 8. Onde este extrator quebra

Três limitações reais, encontradas rodando o próprio código — não hipóteses:

**1. Casamento por substring ignora morfologia.** O mapa tem `pernas inchadas`, o
relato diz "minhas pernas **estão inchadas**". São o mesmo sintoma para qualquer pessoa
e coisas diferentes para o `str.find` — o mesmo acontece com `barriga inchada` vs. "a
barriga está inchada". O remendo é cadastrar a variante à mão (foi o que fizemos com
`desmaio`/`desmaiar`), e é um remendo que não escala: cada verbo conjugado vira uma
linha nova no mapa.

**2. A negação é tratada por janela fixa de 3 palavras.** "Não sinto dor no peito"
funciona; "não sinto, desde que comecei o remédio, nenhuma dor no peito" não.

**3. O mapa é o limite do sistema.** Ele só enxerga as 8 doenças e as ~110 expressões
que nós escrevemos. Um relato de embolia pulmonar ou de dissecção de aorta cai fora do
mapa e sai sem hipótese — o que é preferível a um palpite errado, mas precisa ficar
explícito para quem for usar.

In [9]:
# A limitação nº 1, demonstrada:
for frase in ["minhas pernas estao inchadas", "pernas inchadas",
              "a barriga esta inchada", "barriga inchada",
              "cheguei a desmaiar", "desmaio"]:
    achado = sorted(encontrar_expressoes(frase, expressoes))
    print(f"{frase:<32} -> {achado if achado else '(nada encontrado)'}")

minhas pernas estao inchadas     -> (nada encontrado)
pernas inchadas                  -> ['pernas inchadas']
a barriga esta inchada           -> (nada encontrado)
barriga inchada                  -> ['barriga inchada']
cheguei a desmaiar               -> ['desmaiar']
desmaio                          -> ['desmaio']


---
## 9. O que esta parte entrega

- `data/relatos_pacientes.txt` — 10 relatos completos
- `data/mapa_conhecimento.csv` — 79 regras, 8 doenças, no formato `sintoma_1 | sintoma_2 | doenca_associada`
- `src/extracao.py` — extração + ranqueamento, com negação e evidência explícita
- **10/10** no gabarito interno, com as limitações acima registradas

Na Parte 2 o problema muda: em vez de saber *qual* doença, o sistema precisa decidir
*com que urgência* atender — e lá a regra escrita à mão dá lugar a um modelo treinado.